# 00 — Môi trường thực thi & Khả năng tái lập (Environment & Reproducibility)

## 1. Mục tiêu
Notebook này thiết lập và kiểm toán môi trường tính toán phục vụ toàn bộ dự án **XAI-Football**.
Mục tiêu cụ thể:
- Thu thập metadata về hệ điều hành, phiên bản Python và các thư viện cốt lõi (PyTorch, PyG, scikit-learn, v.v.).
- Kiểm tra tính khả dụng của thiết bị tăng tốc phần cứng (CUDA/GPU).
- Xác minh commit Git HEAD và nhánh làm việc để đảm bảo khả năng truy xuất nguồn gốc (traceability).
- Tính mã băm SHA-256 của toàn bộ file cấu hình trong `configs/` nhằm phát hiện mọi thay đổi cấu hình ngoài ý muốn.
- Kiểm kê toàn diện sự tồn tại của các tập dữ liệu (`raw`, `interim`, `processed`), phân chia (`splits`), mô hình (`models`), và bảng kết quả (`results`).
- Xuất toàn bộ metadata kiểm toán ra file `results/metadata/official/environment.json`.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 0 — Kiểm toán Tiên quyết (Prerequisite Audit)**, được thực thi trước mọi bước tiền xử lý hay huấn luyện mô hình.

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Đảm bảo đường dẫn gốc của repository
REPO_ROOT = Path(".").resolve()
assert (REPO_ROOT / "configs").exists(), "Vui lòng chạy notebook từ thư mục gốc của repository!"

## 3. Input và nguồn dữ liệu
- Môi trường Python runtime hiện tại.
- Các file cấu hình trong thư mục `configs/`.
- Cấu trúc thư mục dữ liệu và artifact: `data/`, `experiments/`, `models/`, `results/`.

## 4. Config/script/module sử dụng
- Các file YAML: `configs/data.yaml`, `configs/graph.yaml`, `configs/split.yaml`, `configs/train.yaml`.
- Module `xai_football.utils.io` và `xai_football.utils.paths`.

## 5. Các bước xử lý
### 5.1. Thu thập phiên bản thư viện và môi trường phần cứng

In [ ]:
import torch

env_info = {
    "os": platform.platform(),
    "python_version": sys.version.split()[0],
    "architecture": platform.machine(),
    "processor": platform.processor(),
}

packages = [
    "torch",
    "torch_geometric",
    "pandas",
    "numpy",
    "scipy",
    "sklearn",
    "matplotlib",
    "seaborn",
    "yaml",
    "nbformat",
]

pkg_versions = {}
for pkg in packages:
    try:
        mod = __import__(pkg)
        pkg_versions[pkg] = getattr(mod, "__version__", "unknown")
    except ImportError:
        pkg_versions[pkg] = "NOT_INSTALLED"

env_info["packages"] = pkg_versions

# Kiểm tra PyTorch & CUDA
env_info["cuda_available"] = torch.cuda.is_available()
env_info["cuda_device_count"] = torch.cuda.device_count()
if torch.cuda.is_available():
    env_info["cuda_device_name"] = torch.cuda.get_device_name(0)
else:
    env_info["cuda_device_name"] = "CPU only"

df_env = pd.DataFrame(list(pkg_versions.items()), columns=["Thư viện", "Phiên bản"])
print(f"Hệ điều hành: {env_info['os']}")
print(f"Python:       {env_info['python_version']}")
print(f"CUDA Device:  {env_info['cuda_device_name']}")
display(df_env)

### 5.2. Kiểm tra Git Branch & Commit SHA

In [ ]:
def get_git_info():
    try:
        branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        status = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
        is_clean = len(status) == 0
        return {"branch": branch, "commit_sha": commit, "clean_working_tree": is_clean}
    except Exception as e:
        return {"error": str(e)}

git_info = get_git_info()
env_info["git"] = git_info
print("Git Branch: ", git_info.get("branch"))
print("Commit SHA: ", git_info.get("commit_sha"))
print("Clean Tree: ", git_info.get("clean_working_tree"))

### 5.3. Băm SHA-256 các file cấu hình (Config Hash Audit)

In [ ]:
def compute_sha256(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

config_hashes = {}
for p in sorted((REPO_ROOT / "configs").glob("**/*.yaml")):
    rel_path = str(p.relative_to(REPO_ROOT)).replace("\\", "/")
    config_hashes[rel_path] = compute_sha256(p)

env_info["config_hashes"] = config_hashes
df_configs = pd.DataFrame(list(config_hashes.items()), columns=["Config File", "SHA-256 Hash"])
display(df_configs)

### 5.4. Kiểm kê Artifact thực tế (Artifact Inventory)

In [ ]:
artifacts_to_check = [
    ("Raw Data Matches (PL 15/16)", REPO_ROOT / "data" / "raw" / "2" / "27"),
    ("Interim Possessions", REPO_ROOT / "data" / "interim" / "official" / "possessions.parquet"),
    ("Interim Passes", REPO_ROOT / "data" / "interim" / "official" / "possession_events.parquet"),
    ("Processed Graphs (.pt)", REPO_ROOT / "data" / "processed" / "official" / "graphs.pt"),
    ("Processed Graph Index", REPO_ROOT / "data" / "processed" / "official" / "graph_index.parquet"),
    ("Train Matches Split", REPO_ROOT / "data" / "splits" / "train_matches.csv"),
    ("Val Matches Split", REPO_ROOT / "data" / "splits" / "val_matches.csv"),
    ("Test Matches Split", REPO_ROOT / "data" / "splits" / "test_matches.csv"),
    ("Graph Split Summary", REPO_ROOT / "results" / "tables" / "official" / "graph_split_summary.csv"),
    ("Official Model Comparison", REPO_ROOT / "results" / "tables" / "official" / "model_comparison.csv"),
    ("Official Baseline Comparison", REPO_ROOT / "results" / "tables" / "official" / "baseline_comparison.csv"),
    ("MPNN Best Checkpoint", REPO_ROOT / "models" / "official" / "mpnn_best.pt"),
]

inventory_rows = []
for label, path in artifacts_to_check:
    exists = path.exists()
    size_str = "N/A"
    if exists:
        if path.is_file():
            size_mb = path.stat().st_size / (1024 * 1024)
            size_str = f"{size_mb:.2f} MB"
        else:
            n_items = len(list(path.iterdir()))
            size_str = f"{n_items} files"
    inventory_rows.append({"Artifact": label, "Tồn tại": exists, "Đường dẫn": str(path.relative_to(REPO_ROOT)), "Kích thước": size_str})

df_inventory = pd.DataFrame(inventory_rows)
display(df_inventory)

## 6. Kết quả và cách diễn giải
- Bảng trên liệt kê tình trạng sẵn sàng của các artifact chính.
- Nếu các artifact cốt lõi (`possessions.parquet`, `graphs.pt`, `graph_split_summary.csv`) đều tồn tại (`Tồn tại = True`), toàn bộ pipeline đã được chuẩn bị đầy đủ để tái lập các phân tích và biểu đồ trực quan.

## 7. Kiểm tra chất lượng (Quality Checks)

In [ ]:
checks = []
checks.append({"Check": "Python Version >= 3.10", "Status": "PASS" if sys.version_info >= (3, 10) else "FAIL"})
checks.append({"Check": "PyTorch Installed", "Status": "PASS" if pkg_versions.get("torch") != "NOT_INSTALLED" else "FAIL"})
checks.append({"Check": "PyG Installed", "Status": "PASS" if pkg_versions.get("torch_geometric") != "NOT_INSTALLED" else "FAIL"})
checks.append({"Check": "Config Hashes Computed", "Status": "PASS" if len(config_hashes) >= 4 else "FAIL"})

df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["Status"] == "PASS").all(), "Có kiểm tra chất lượng môi trường không đạt!"

## 8. Output được lưu ở đâu

In [ ]:
out_meta_dir = REPO_ROOT / "results" / "metadata" / "official"
out_meta_dir.mkdir(parents=True, exist_ok=True)
env_out_path = out_meta_dir / "environment.json"

with open(env_out_path, "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=2, ensure_ascii=False)

print(f"Đã lưu thông tin môi trường vào: {env_out_path}")

## 9. Cách tái chạy
Chạy lệnh sau từ thư mục gốc của repository:
```bash
pip install -e .
python -m jupyter nbconvert --to notebook --execute notebooks/00_environment_and_reproducibility.ipynb
```

## 10. Giới hạn và giả định
- Notebook phụ thuộc vào môi trường đã cài các gói trong `requirements.txt`.
- Kết quả kiểm tra thiết bị CUDA phụ thuộc vào máy của người dùng (nếu không có GPU NVIDIA, code tự động chuyển về CPU an toàn).